# Benchmark Comparativo: FAISS × Milvus × Chroma em Pipelines RAG

**Trabalho de Conclusão de Curso — Análise e Desenvolvimento de Sistemas**  
**Autor:** Breno Alves  
**Título:** Comparação de Bancos de Vetores aplicados a Soluções de Geração Aumentada de Recuperação (RAG)

---

## Objetivos deste notebook

Este notebook implementa a metodologia experimental descrita na monografia, comparando os três principais bancos de vetores de código aberto — **FAISS**, **Milvus** e **Chroma** — em três cenários de escala distintos:

| Cenário | Volume | Objetivo |
|---------|--------|----------|
| **S1** | ~10.000 vetores | Comportamento em prototipagem e aplicações pequenas |
| **S2** | ~100.000 vetores | Escala intermediária / produção leve |
| **S3** | ~1.000.000 vetores | Escala empresarial / limite de viabilidade |

### Métricas coletadas
- **Tempo de indexação** (ingestão + construção do índice)
- **Latência de busca** (p50, p90, p99 em ms)
- **Throughput** (queries por segundo — QPS)
- **Recall@k** (k=1, 5, 10) — qualidade de recuperação semântica
- **Consumo de memória RAM** (pico durante indexação e consulta)
- **Tamanho em disco** do índice persistido

### Estratégia de embeddings
- **S1 (10k):** embeddings reais gerados com `sentence-transformers/all-MiniLM-L6-v2` (384 dims) sobre corpus textual
- **S2 (100k) e S3 (1M):** vetores sintéticos normalizados de mesma dimensionalidade (384 dims), garantindo reprodutibilidade e viabilidade temporal do experimento. Esta abordagem é metodologicamente válida para avaliar desempenho de indexação e busca, isolando o componente vetorial do pipeline.

> **Nota metodológica:** A geração de ground truth para Recall@k utiliza busca exata (FAISS IndexFlatL2) como oráculo, prática padrão em benchmarks de ANN (Approximate Nearest Neighbor).

## 0. Instalação de Dependências

In [ ]:
# Execute este bloco apenas na primeira execução
# Tempo estimado: 3–5 minutos
%pip install -r requirements.txt

print("✅ Dependências instaladas com sucesso.")

## 1. Imports e Configuração Global

In [1]:
import os
import gc
import time
import shutil
import warnings
import numpy as np
import pandas as pd
import psutil
import faiss
import chromadb
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from tqdm.notebook import tqdm
from pathlib import Path
from sentence_transformers import SentenceTransformer
from pymilvus import MilvusClient
from typing import List, Dict, Tuple, Optional
from contextlib import contextmanager

warnings.filterwarnings("ignore")

# ─── Configuração reprodutível ────────────────────────────────────────────────
SEED        = 42
np.random.seed(SEED)

# ─── Parâmetros dos cenários ──────────────────────────────────────────────────
EMBEDDING_DIM = 384           # dimensão do modelo all-MiniLM-L6-v2
SCENARIO_SIZES = {
    "S1_10k":  10_000,
    "S2_100k": 100_000,
    "S3_1M":   1_000_000,
}
TOP_K_VALUES  = [1, 5, 10]    # valores de k para Recall@k
N_QUERIES     = 200           # consultas por cenário
N_RUNS        = 5             # repetições para estabilidade estatística

# ─── Diretórios de trabalho ────────────────────────────────────────────────────
WORK_DIR     = Path("./benchmark_output")
CHROMA_DIR   = WORK_DIR / "chroma_db"
MILVUS_DB    = WORK_DIR / "milvus_lite.db"
FAISS_DIR    = WORK_DIR / "faiss_indices"
WORK_DIR.mkdir(exist_ok=True)
FAISS_DIR.mkdir(exist_ok=True)

# ─── Paleta de cores para gráficos ────────────────────────────────────────────
COLORS = {
    "FAISS":  "#3B82F6",   # azul
    "Milvus": "#10B981",   # verde
    "Chroma": "#F59E0B",   # âmbar
}

plt.rcParams.update({
    "figure.dpi":         150,
    "axes.spines.top":    False,
    "axes.spines.right":  False,
    "font.family":        "DejaVu Sans",
})

print(f"✅ Configuração carregada | dim={EMBEDDING_DIM} | seed={SEED}")
print(f"   Cenários: {list(SCENARIO_SIZES.keys())}")
print(f"   Queries por cenário: {N_QUERIES} | Repetições: {N_RUNS}")

/home/breno-oliveira/Documentos/gitRepositories/AulasFatec/tcc/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Configuração carregada | dim=384 | seed=42
   Cenários: ['S1_10k', 'S2_100k', 'S3_1M']
   Queries por cenário: 200 | Repetições: 5


## 2. Utilitários de Medição

In [2]:
def get_memory_mb() -> float:
    """Retorna uso de RAM do processo atual em MB."""
    proc = psutil.Process(os.getpid())
    return proc.memory_info().rss / (1024 ** 2)


@contextmanager
def measure_memory():
    """Context manager que captura o delta de memória RAM (MB) durante o bloco."""
    gc.collect()
    mem_before = get_memory_mb()
    stats = {"peak_delta_mb": 0.0, "baseline_mb": mem_before}
    try:
        yield stats
    finally:
        gc.collect()
        mem_after = get_memory_mb()
        stats["peak_delta_mb"] = max(0.0, mem_after - mem_before)


def measure_latency_percentiles(
    query_fn,
    queries: np.ndarray,
    n_runs: int = N_RUNS,
) -> Dict[str, float]:
    """
    Executa `query_fn(queries)` `n_runs` vezes e retorna estatísticas
    de latência por query em milissegundos.

    Descarta a primeira execução (warm-up de cache/JIT).
    """
    latencies_ms = []
    for i in range(n_runs + 1):          # +1 para warm-up
        start = time.perf_counter()
        query_fn(queries)
        elapsed = (time.perf_counter() - start) * 1000
        if i > 0:                        # descarta warm-up
            latencies_ms.append(elapsed / len(queries))   # ms por query

    arr = np.array(latencies_ms)
    return {
        "p50_ms":      float(np.percentile(arr, 50)),
        "p90_ms":      float(np.percentile(arr, 90)),
        "p99_ms":      float(np.percentile(arr, 99)),
        "mean_ms":     float(arr.mean()),
        "qps":         float(1000 / arr.mean()),   # queries por segundo
    }


def compute_recall_at_k(
    retrieved: np.ndarray,    # (n_queries, k) — índices retornados pelo banco
    ground_truth: np.ndarray, # (n_queries, k_max) — índices reais (oráculo)
    k: int,
) -> float:
    """
    Recall@k = fração de queries onde pelo menos 1 resultado verdadeiro
    está entre os k recuperados. Métrica padrão em benchmarks ANN.
    """
    hits = 0
    for i in range(len(retrieved)):
        true_set = set(ground_truth[i, :k])
        pred_set = set(retrieved[i, :k])
        if true_set & pred_set:
            hits += 1
    return hits / len(retrieved)


def get_dir_size_mb(path: Path) -> float:
    """Retorna tamanho em MB de um diretório ou arquivo."""
    if path.is_file():
        return path.stat().st_size / (1024 ** 2)
    if path.is_dir():
        return sum(f.stat().st_size for f in path.rglob("*") if f.is_file()) / (1024 ** 2)
    return 0.0


print("✅ Utilitários de medição definidos.")

✅ Utilitários de medição definidos.


## 3. Geração de Dados

### 3.1 Cenário S1 — Embeddings Reais (10k)

Para o cenário de menor escala, utilizamos o dataset **ag_news** (notícias) com o modelo **`all-MiniLM-L6-v2`**, gerando embeddings semânticos reais. Isso permite avaliar a qualidade de recuperação em condições próximas a um pipeline RAG de produção.

In [3]:
from datasets import load_dataset

N_REAL = SCENARIO_SIZES["S1_10k"]

print(f"⏳ Carregando corpus textual (ag_news, {N_REAL:,} amostras)...")
ds = load_dataset("ag_news", split="train", trust_remote_code=True)
texts_10k = [row["text"] for row in ds.select(range(N_REAL))]

print(f"⏳ Carregando modelo de embeddings: all-MiniLM-L6-v2...")
embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

print(f"⏳ Gerando {N_REAL:,} embeddings reais (pode levar 2–3 min)...")
with measure_memory() as mem_embed:
    embeddings_10k = embedder.encode(
        texts_10k,
        batch_size=256,
        show_progress_bar=True,
        normalize_embeddings=True,   # normalização para cosine similarity
    ).astype(np.float32)

# Gera queries reais a partir de amostras separadas do corpus
query_texts_10k  = [row["text"] for row in ds.select(range(N_REAL, N_REAL + N_QUERIES))]
query_vecs_10k   = embedder.encode(
    query_texts_10k, batch_size=256, normalize_embeddings=True
).astype(np.float32)

print(f"\n✅ S1 | corpus={embeddings_10k.shape} | queries={query_vecs_10k.shape}")
print(f"   RAM consumida na geração: {mem_embed['peak_delta_mb']:.1f} MB")

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'ag_news' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


⏳ Carregando corpus textual (ag_news, 10,000 amostras)...


HfUriError: Invalid HF URI 'hf://datasets/ag_news@eb185aade064a813bc0b7f42de02595523103ca4/.huggingface.yaml'. Repository id must be 'namespace/name', got 'ag_news'.

### 3.2 Cenários S2 e S3 — Vetores Sintéticos Normalizados (100k e 1M)

In [ ]:
def generate_synthetic_vectors(n: int, dim: int, seed: int = SEED) -> np.ndarray:
    """
    Gera n vetores aleatórios normalizados em R^dim.
    A normalização garante que a busca por produto interno seja equivalente
    à similaridade de cosseno, replicando o comportamento de embeddings reais.
    """
    rng  = np.random.default_rng(seed)
    vecs = rng.standard_normal((n, dim)).astype(np.float32)
    norms = np.linalg.norm(vecs, axis=1, keepdims=True)
    return vecs / norms


print("⏳ Gerando vetores sintéticos para S2 (100k)...")
embeddings_100k  = generate_synthetic_vectors(SCENARIO_SIZES["S2_100k"], EMBEDDING_DIM)
query_vecs_100k  = generate_synthetic_vectors(N_QUERIES, EMBEDDING_DIM, seed=SEED + 1)

print("⏳ Gerando vetores sintéticos para S3 (1M)...")
# 1M × 384 × 4 bytes ≈ 1.44 GB — gerado em blocos se necessário
embeddings_1M    = generate_synthetic_vectors(SCENARIO_SIZES["S3_1M"], EMBEDDING_DIM)
query_vecs_1M    = generate_synthetic_vectors(N_QUERIES, EMBEDDING_DIM, seed=SEED + 2)

print(f"\n✅ S2 | corpus={embeddings_100k.shape} | {embeddings_100k.nbytes / 1e6:.1f} MB")
print(f"✅ S3 | corpus={embeddings_1M.shape}    | {embeddings_1M.nbytes / 1e6:.0f} MB")

### 3.3 Ground Truth — Oráculo de Busca Exata

O ground truth para Recall@k é computado via **FAISS IndexFlatIP** (produto interno exato), que retorna os k vizinhos mais próximos sem aproximação. Esta é a prática padrão do ANN-Benchmarks.

In [ ]:
def compute_ground_truth(
    corpus: np.ndarray,
    queries: np.ndarray,
    k: int = max(TOP_K_VALUES),
) -> np.ndarray:
    """
    Busca exata via FAISS IndexFlatIP (Inner Product = cosine em vetores normalizados).
    Retorna array (n_queries, k) com os índices verdadeiros.
    """
    index = faiss.IndexFlatIP(corpus.shape[1])
    index.add(corpus)
    _, I = index.search(queries, k)
    return I


print("⏳ Computando ground truth para S1 (10k)...")
gt_10k  = compute_ground_truth(embeddings_10k, query_vecs_10k)

print("⏳ Computando ground truth para S2 (100k)...")
gt_100k = compute_ground_truth(embeddings_100k, query_vecs_100k)

print("⏳ Computando ground truth para S3 (1M) — pode demorar ~30s...")
gt_1M   = compute_ground_truth(embeddings_1M, query_vecs_1M)

print("\n✅ Ground truth computado para todos os cenários.")

## 4. Benchmark — FAISS

O FAISS é avaliado com dois índices distintos para capturar o trade-off entre precisão e desempenho:
- **`IndexFlatIP`** — busca exata (baseline de recall = 1.0)
- **`IndexHNSWFlat`** — busca aproximada com grafo hierárquico (HNSW), configuração padrão de produção
- **`IndexIVFFlat`** — busca aproximada com Inverted File Index, favorável em 1M+

In [ ]:
def benchmark_faiss(
    corpus: np.ndarray,
    queries: np.ndarray,
    ground_truth: np.ndarray,
    scenario_name: str,
    k: int = 10,
) -> Dict:
    """
    Executa benchmark completo do FAISS com múltiplos índices.
    Retorna dicionário de resultados por configuração de índice.
    """
    results = {}
    n, dim = corpus.shape

    # Configurações de índice a testar
    index_configs = {
        "Flat (exato)": lambda: faiss.IndexFlatIP(dim),
        "HNSW-32":      lambda: faiss.IndexHNSWFlat(dim, 32),    # M=32 conexões
    }

    # Para volumes grandes, adiciona IVF
    if n >= 100_000:
        nlist = int(np.sqrt(n))          # regra empírica: sqrt(n) células
        def build_ivf():
            quantizer = faiss.IndexFlatIP(dim)
            return faiss.IndexIVFFlat(quantizer, dim, nlist, faiss.METRIC_INNER_PRODUCT)
        index_configs[f"IVF{nlist}"] = build_ivf

    for idx_name, build_fn in index_configs.items():
        print(f"  ├─ [{scenario_name}] FAISS {idx_name}")
        index = build_fn()

        # ── Indexação ─────────────────────────────────────────────────────────
        with measure_memory() as mem_idx:
            t0 = time.perf_counter()
            if hasattr(index, "train") and not index.is_trained:
                index.train(corpus)
            index.add(corpus)
            index_time_s = time.perf_counter() - t0

        # ── Persistência ──────────────────────────────────────────────────────
        idx_path = FAISS_DIR / f"{scenario_name}_{idx_name.replace(' ','_')}.index"
        faiss.write_index(index, str(idx_path))
        disk_mb = get_dir_size_mb(idx_path)

        # ── Latência e Throughput ──────────────────────────────────────────────
        def query_fn(q):
            _, I = index.search(q, k)
            return I

        lat_stats = measure_latency_percentiles(query_fn, queries)

        # ── Recall@k ──────────────────────────────────────────────────────────
        _, I_retrieved = index.search(queries, k)
        recall_scores = {
            f"recall@{kv}": compute_recall_at_k(I_retrieved, ground_truth, kv)
            for kv in TOP_K_VALUES
        }

        results[idx_name] = {
            "index_time_s":   index_time_s,
            "ram_delta_mb":   mem_idx["peak_delta_mb"],
            "disk_mb":        disk_mb,
            **lat_stats,
            **recall_scores,
        }

    return results


faiss_results = {}

print("\n🔵 Benchmarking FAISS...")
for scenario, (corpus, queries, gt) in [
    ("S1_10k",  (embeddings_10k,  query_vecs_10k,  gt_10k)),
    ("S2_100k", (embeddings_100k, query_vecs_100k, gt_100k)),
    ("S3_1M",   (embeddings_1M,   query_vecs_1M,   gt_1M)),
]:
    print(f"\n  📦 Cenário {scenario} ({corpus.shape[0]:,} vetores)")
    faiss_results[scenario] = benchmark_faiss(corpus, queries, gt, scenario)

print("\n✅ FAISS benchmark concluído.")

## 5. Benchmark — Milvus (Milvus Lite)

Utilizamos **Milvus Lite** (modo embarcado via `pymilvus`), que executa localmente sem necessidade de Docker, mantendo a mesma API e algoritmos de indexação do Milvus completo. Esta abordagem é adequada para fins de benchmark e comparação metodológica.

In [ ]:
def benchmark_milvus(
    corpus: np.ndarray,
    queries: np.ndarray,
    ground_truth: np.ndarray,
    scenario_name: str,
    k: int = 10,
) -> Dict:
    """
    Benchmark Milvus Lite com índices FLAT e HNSW.
    Cada execução usa uma base de dados isolada para garantir limpeza de estado.
    """
    results = {}
    n, dim = corpus.shape

    index_configs = [
        {
            "name": "Flat (exato)",
            "index_type": "FLAT",
            "metric_type": "IP",
            "params": {},
            "search_params": {},
        },
        {
            "name": "HNSW-16",
            "index_type": "HNSW",
            "metric_type": "IP",
            "params": {"M": 16, "efConstruction": 200},
            "search_params": {"ef": 64},
        },
    ]

    if n >= 100_000:
        index_configs.append({
            "name": "IVF_FLAT",
            "index_type": "IVF_FLAT",
            "metric_type": "IP",
            "params": {"nlist": int(np.sqrt(n))},
            "search_params": {"nprobe": 32},
        })

    for cfg in index_configs:
        idx_name = cfg["name"]
        print(f"  ├─ [{scenario_name}] Milvus {idx_name}")

        db_path = str(WORK_DIR / f"milvus_{scenario_name}_{idx_name.replace(' ','_')}.db")
        if os.path.exists(db_path):
            os.remove(db_path)

        client = MilvusClient(db_path)
        coll_name = "benchmark"

        # ── Indexação ─────────────────────────────────────────────────────────
        with measure_memory() as mem_idx:
            t0 = time.perf_counter()

            client.create_collection(
                collection_name=coll_name,
                dimension=dim,
                metric_type=cfg["metric_type"],
                index_type=cfg["index_type"],
                index_params=cfg["params"],
            )

            # Inserção em batches de 5.000 para evitar overhead de payload
            BATCH = 5_000
            for start in range(0, n, BATCH):
                batch_vecs = corpus[start:start + BATCH]
                data = [
                    {"id": int(start + j), "vector": batch_vecs[j].tolist()}
                    for j in range(len(batch_vecs))
                ]
                client.insert(collection_name=coll_name, data=data)

            index_time_s = time.perf_counter() - t0

        disk_mb = get_dir_size_mb(Path(db_path))

        # ── Latência e Throughput ──────────────────────────────────────────────
        search_params = cfg["search_params"]

        def query_fn(q):
            return client.search(
                collection_name=coll_name,
                data=q.tolist(),
                limit=k,
                search_params=search_params,
            )

        lat_stats = measure_latency_percentiles(query_fn, queries)

        # ── Recall@k ──────────────────────────────────────────────────────────
        raw = client.search(
            collection_name=coll_name,
            data=queries.tolist(),
            limit=k,
            search_params=search_params,
        )
        I_retrieved = np.array([[hit["id"] for hit in res] for res in raw])
        recall_scores = {
            f"recall@{kv}": compute_recall_at_k(I_retrieved, ground_truth, kv)
            for kv in TOP_K_VALUES
        }

        client.close()

        results[idx_name] = {
            "index_time_s": index_time_s,
            "ram_delta_mb": mem_idx["peak_delta_mb"],
            "disk_mb":      disk_mb,
            **lat_stats,
            **recall_scores,
        }

    return results


milvus_results = {}

print("\n🟢 Benchmarking Milvus...")
for scenario, (corpus, queries, gt) in [
    ("S1_10k",  (embeddings_10k,  query_vecs_10k,  gt_10k)),
    ("S2_100k", (embeddings_100k, query_vecs_100k, gt_100k)),
    ("S3_1M",   (embeddings_1M,   query_vecs_1M,   gt_1M)),
]:
    print(f"\n  📦 Cenário {scenario} ({corpus.shape[0]:,} vetores)")
    milvus_results[scenario] = benchmark_milvus(corpus, queries, gt, scenario)

print("\n✅ Milvus benchmark concluído.")

## 6. Benchmark — Chroma

O Chroma é avaliado em modo **persistente** com backend HNSW (padrão interno). Para o cenário S3 (1M), o Chroma é testado até onde operacionalmente viável, registrando-se degradações de desempenho quando observadas — dado relevante para a comparação.

In [ ]:
def benchmark_chroma(
    corpus: np.ndarray,
    queries: np.ndarray,
    ground_truth: np.ndarray,
    scenario_name: str,
    k: int = 10,
) -> Dict:
    """
    Benchmark do ChromaDB com índice HNSW (único tipo disponível nativamente).
    O Chroma não expõe seleção de tipo de índice — usa HNSW com configuração
    automática, o que é relevante como achado comparativo.
    """
    n, dim = corpus.shape
    coll_path = WORK_DIR / f"chroma_{scenario_name}"
    if coll_path.exists():
        shutil.rmtree(coll_path)

    # Aviso de viabilidade para S3
    if n >= 1_000_000:
        print(f"  ⚠️  [{scenario_name}] Chroma com 1M vetores: operação lenta esperada.")
        print(f"      A inserção será feita em batches; o tempo reflete limitação arquitetural.")

    client = chromadb.PersistentClient(path=str(coll_path))
    coll_name = f"benchmark_{scenario_name}"

    # ── Indexação ──────────────────────────────────────────────────────────────
    with measure_memory() as mem_idx:
        t0 = time.perf_counter()
        collection = client.create_collection(
            name=coll_name,
            metadata={"hnsw:space": "cosine"},
        )

        BATCH = 5_000   # Chroma impõe limite de batch
        for start in tqdm(range(0, n, BATCH), desc=f"  Chroma ingest {scenario_name}", leave=False):
            batch_vecs = corpus[start:start + BATCH]
            batch_size = len(batch_vecs)
            collection.add(
                ids=[str(start + j) for j in range(batch_size)],
                embeddings=batch_vecs.tolist(),
            )

        index_time_s = time.perf_counter() - t0

    disk_mb = get_dir_size_mb(coll_path)

    # ── Latência e Throughput ──────────────────────────────────────────────────
    def query_fn(q):
        return collection.query(
            query_embeddings=q.tolist(),
            n_results=k,
            include=[],   # retorna apenas IDs, sem overhead de metadados
        )

    lat_stats = measure_latency_percentiles(query_fn, queries)

    # ── Recall@k ──────────────────────────────────────────────────────────────
    raw = collection.query(
        query_embeddings=queries.tolist(),
        n_results=k,
        include=[],
    )
    I_retrieved = np.array([
        [int(idx) for idx in res]
        for res in raw["ids"]
    ])
    recall_scores = {
        f"recall@{kv}": compute_recall_at_k(I_retrieved, ground_truth, kv)
        for kv in TOP_K_VALUES
    }

    return {
        "HNSW (padrão)": {
            "index_time_s": index_time_s,
            "ram_delta_mb": mem_idx["peak_delta_mb"],
            "disk_mb":      disk_mb,
            **lat_stats,
            **recall_scores,
        }
    }


chroma_results = {}

print("\n🟡 Benchmarking Chroma...")
for scenario, (corpus, queries, gt) in [
    ("S1_10k",  (embeddings_10k,  query_vecs_10k,  gt_10k)),
    ("S2_100k", (embeddings_100k, query_vecs_100k, gt_100k)),
    ("S3_1M",   (embeddings_1M,   query_vecs_1M,   gt_1M)),
]:
    print(f"\n  📦 Cenário {scenario} ({corpus.shape[0]:,} vetores)")
    chroma_results[scenario] = benchmark_chroma(corpus, queries, gt, scenario)

print("\n✅ Chroma benchmark concluído.")

## 7. Consolidação dos Resultados

In [ ]:
def consolidate_results(faiss_r, milvus_r, chroma_r) -> pd.DataFrame:
    """
    Consolida todos os resultados em um DataFrame tabular.
    Para comparações justas entre bancos, seleciona a melhor configuração
    de cada banco por cenário (critério: melhor Recall@10 entre os índices ANN).
    """
    rows = []

    def best_ann_config(results_dict):
        """Seleciona configuração com melhor recall@10, excluindo busca exata."""
        ann_entries = {
            k: v for k, v in results_dict.items()
            if "exato" not in k.lower() and "flat" not in k.lower()
        } or results_dict   # fallback para Flat se só tiver esse
        return max(ann_entries.items(), key=lambda x: x[1].get("recall@10", 0))

    for scenario in ["S1_10k", "S2_100k", "S3_1M"]:
        n_vecs = SCENARIO_SIZES[scenario.replace("S1_","S1_").replace("S2_","S2_").replace("S3_","S3_")]

        db_map = {
            "FAISS":  faiss_r.get(scenario, {}),
            "Milvus": milvus_r.get(scenario, {}),
            "Chroma": chroma_r.get(scenario, {}),
        }

        for db_name, db_data in db_map.items():
            if not db_data:
                continue

            # Para Chroma, só existe HNSW; para outros, pega melhor ANN
            if db_name == "Chroma":
                cfg_name, metrics = list(db_data.items())[0]
            else:
                cfg_name, metrics = best_ann_config(db_data)

            rows.append({
                "Banco":          db_name,
                "Cenário":        scenario,
                "N_vetores":      n_vecs,
                "Índice":         cfg_name,
                "Indexação (s)":  round(metrics.get("index_time_s", np.nan), 2),
                "RAM (MB)":       round(metrics.get("ram_delta_mb", np.nan), 1),
                "Disco (MB)":     round(metrics.get("disk_mb", np.nan), 1),
                "Latência p50 (ms)": round(metrics.get("p50_ms", np.nan), 3),
                "Latência p99 (ms)": round(metrics.get("p99_ms", np.nan), 3),
                "QPS":            round(metrics.get("qps", np.nan), 1),
                "Recall@1":       round(metrics.get("recall@1", np.nan), 4),
                "Recall@5":       round(metrics.get("recall@5", np.nan), 4),
                "Recall@10":      round(metrics.get("recall@10", np.nan), 4),
            })

    return pd.DataFrame(rows)


df = consolidate_results(faiss_results, milvus_results, chroma_results)

# Exibe a tabela completa
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)
pd.set_option("display.float_format", "{:.3f}".format)
print("\n=== RESULTADOS CONSOLIDADOS ===\n")
display(df.set_index(["Banco", "Cenário"]).drop(columns=["N_vetores", "Índice"]))

## 8. Visualizações

### 8.1 Tempo de Indexação × Escala

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

metrics_to_plot = [
    ("Indexação (s)",    "Tempo de Indexação (s)",    "Tempo de Indexação por Cenário"),
    ("Latência p50 (ms)","Latência p50 por Query (ms)","Latência Mediana de Busca"),
    ("QPS",              "Queries por Segundo (QPS)",  "Throughput — Queries por Segundo"),
]

scenarios   = ["S1_10k", "S2_100k", "S3_1M"]
x           = np.arange(len(scenarios))
width       = 0.25

for ax, (metric, ylabel, title) in zip(axes, metrics_to_plot):
    for i, db in enumerate(["FAISS", "Milvus", "Chroma"]):
        vals = [
            df[(df["Banco"] == db) & (df["Cenário"] == s)][metric].values
            for s in scenarios
        ]
        vals = [v[0] if len(v) > 0 else np.nan for v in vals]
        bars = ax.bar(x + i * width, vals, width, label=db,
                      color=COLORS[db], alpha=0.87, edgecolor="white", linewidth=0.8)

        # Anotação de valor nas barras
        for bar, v in zip(bars, vals):
            if not np.isnan(v):
                ax.text(
                    bar.get_x() + bar.get_width() / 2,
                    bar.get_height() * 1.02,
                    f"{v:.1f}" if v >= 1 else f"{v:.3f}",
                    ha="center", va="bottom", fontsize=7, fontweight="bold"
                )

    ax.set_title(title, fontsize=11, fontweight="bold", pad=10)
    ax.set_ylabel(ylabel, fontsize=9)
    ax.set_xticks(x + width)
    ax.set_xticklabels(["10k", "100k", "1M"], fontsize=10)
    ax.set_xlabel("Volume de Vetores", fontsize=9)
    ax.legend(fontsize=8)
    ax.yaxis.grid(True, linestyle="--", alpha=0.5)
    ax.set_axisbelow(True)

fig.suptitle(
    "FAISS × Milvus × Chroma — Desempenho por Cenário de Escala",
    fontsize=13, fontweight="bold", y=1.02
)
plt.tight_layout()
plt.savefig(WORK_DIR / "fig1_desempenho_escala.png", bbox_inches="tight", dpi=200)
plt.show()
print("✅ Figura 1 salva.")

### 8.2 Recall@k — Qualidade de Recuperação Semântica

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=True)

for ax, scenario in zip(axes, scenarios):
    subset = df[df["Cenário"] == scenario]
    k_labels = ["@1", "@5", "@10"]
    k_cols   = ["Recall@1", "Recall@5", "Recall@10"]
    x_k      = np.arange(len(k_labels))

    for i, db in enumerate(["FAISS", "Milvus", "Chroma"]):
        row = subset[subset["Banco"] == db]
        if row.empty:
            continue
        vals = [row[c].values[0] for c in k_cols]
        ax.plot(
            k_labels, vals,
            marker="o", linewidth=2.2, markersize=8,
            color=COLORS[db], label=db
        )
        for xi, (kl, v) in enumerate(zip(k_labels, vals)):
            ax.annotate(
                f"{v:.3f}",
                (kl, v), textcoords="offset points",
                xytext=(0, 8), ha="center", fontsize=8, color=COLORS[db]
            )

    ax.set_title(f"Cenário {scenario}", fontsize=11, fontweight="bold")
    ax.set_xlabel("k", fontsize=10)
    ax.set_ylim(0, 1.12)
    ax.set_yticks(np.arange(0, 1.1, 0.1))
    ax.yaxis.grid(True, linestyle="--", alpha=0.5)
    ax.set_axisbelow(True)
    ax.legend(fontsize=9)

axes[0].set_ylabel("Recall@k", fontsize=10)

fig.suptitle(
    "Recall@k — Qualidade de Recuperação Semântica por Cenário",
    fontsize=13, fontweight="bold", y=1.02
)
plt.tight_layout()
plt.savefig(WORK_DIR / "fig2_recall_k.png", bbox_inches="tight", dpi=200)
plt.show()
print("✅ Figura 2 salva.")

### 8.3 Heatmap — Consumo de Recursos

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for ax, metric, title, fmt in [
    (axes[0], "RAM (MB)",    "Consumo de RAM na Indexação (MB)",   ".0f"),
    (axes[1], "Disco (MB)",  "Tamanho do Índice em Disco (MB)",    ".0f"),
]:
    pivot = df.pivot_table(
        index="Banco", columns="Cenário", values=metric, aggfunc="first"
    )[["S1_10k", "S2_100k", "S3_1M"]]   # garante ordem

    sns.heatmap(
        pivot, ax=ax, annot=True, fmt=fmt,
        cmap="YlOrRd", linewidths=0.5,
        cbar_kws={"shrink": 0.8},
        annot_kws={"size": 11, "weight": "bold"}
    )
    ax.set_title(title, fontsize=11, fontweight="bold", pad=12)
    ax.set_xlabel("Cenário", fontsize=10)
    ax.set_ylabel("Banco de Vetores", fontsize=10)
    ax.set_xticklabels(["10k", "100k", "1M"], fontsize=10)

plt.tight_layout()
plt.savefig(WORK_DIR / "fig3_recursos.png", bbox_inches="tight", dpi=200)
plt.show()
print("✅ Figura 3 salva.")

### 8.4 Curva de Latência — Distribuição Percentílica

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=False)

percentile_cols = ["Latência p50 (ms)", "Latência p99 (ms)"]

for ax, scenario in zip(axes, scenarios):
    subset = df[df["Cenário"] == scenario]
    n_vecs = SCENARIO_SIZES[scenario.replace("S1_","S1_").replace("S2_","S2_").replace("S3_","S3_")]

    db_names = ["FAISS", "Milvus", "Chroma"]
    x = np.arange(len(db_names))
    width = 0.35

    for j, (col, label, alpha) in enumerate([
        ("Latência p50 (ms)", "p50 (mediana)", 1.0),
        ("Latência p99 (ms)", "p99 (cauda)",   0.55),
    ]):
        vals = []
        colors = []
        for db in db_names:
            row = subset[subset["Banco"] == db]
            v = row[col].values[0] if not row.empty else np.nan
            vals.append(v)
            colors.append(COLORS[db])

        offset = (j - 0.5) * width
        bars = ax.bar(x + offset, vals, width * 0.9, label=label,
                      color=colors, alpha=alpha, edgecolor="white", linewidth=0.8)

        for bar, v in zip(bars, vals):
            if not np.isnan(v):
                ax.text(
                    bar.get_x() + bar.get_width() / 2,
                    bar.get_height() * 1.02,
                    f"{v:.3f}",
                    ha="center", va="bottom", fontsize=7, fontweight="bold"
                )

    ax.set_title(f"{scenario}\n({n_vecs:,} vetores)", fontsize=10, fontweight="bold")
    ax.set_ylabel("Latência (ms) por query", fontsize=9)
    ax.set_xticks(x)
    ax.set_xticklabels(db_names, fontsize=10)

    # Legenda de padrão (sólido = p50, semitransparente = p99)
    patch_p50 = mpatches.Patch(facecolor="gray", alpha=1.0, label="p50 (mediana)")
    patch_p99 = mpatches.Patch(facecolor="gray", alpha=0.55, label="p99 (cauda)")
    ax.legend(handles=[patch_p50, patch_p99], fontsize=8)
    ax.yaxis.grid(True, linestyle="--", alpha=0.5)
    ax.set_axisbelow(True)

fig.suptitle(
    "Distribuição de Latência por Query (p50 e p99) — ms",
    fontsize=13, fontweight="bold", y=1.02
)
plt.tight_layout()
plt.savefig(WORK_DIR / "fig4_latencia_percentis.png", bbox_inches="tight", dpi=200)
plt.show()
print("✅ Figura 4 salva.")

### 8.5 Gráfico Radar — Análise Multidimensional por Banco

In [ ]:
from matplotlib.patches import FancyArrowPatch

def normalize_col(series: pd.Series, higher_is_better: bool) -> pd.Series:
    mn, mx = series.min(), series.max()
    if mx == mn:
        return pd.Series(0.5, index=series.index)
    norm = (series - mn) / (mx - mn)
    return norm if higher_is_better else (1 - norm)


# Dimensões do radar e sua direção (maior = melhor?)
RADAR_DIMS = [
    ("QPS",               True,   "Throughput (QPS)"),
    ("Recall@10",         True,   "Recall@10"),
    ("Latência p99 (ms)", False,  "Baixa Latência p99"),
    ("RAM (MB)",          False,  "Eficiência RAM"),
    ("Disco (MB)",        False,  "Eficiência Disco"),
    ("Indexação (s)",     False,  "Velocidade Indexação"),
]

# Agrega métricas por banco (média entre cenários, excluindo S3 para normalização justa)
df_agg = df[df["Cenário"] != "S3_1M"].groupby("Banco").mean(numeric_only=True).reset_index()

for col, hib, _ in RADAR_DIMS:
    df_agg[f"{col}_norm"] = normalize_col(df_agg[col], hib)

norm_cols = [f"{col}_norm" for col, _, __ in RADAR_DIMS]
labels    = [lbl for _, __, lbl in RADAR_DIMS]
N         = len(RADAR_DIMS)
angles    = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
angles   += angles[:1]   # fecha o polígono

fig, ax = plt.subplots(figsize=(7, 7), subplot_kw=dict(polar=True))

for _, row in df_agg.iterrows():
    db   = row["Banco"]
    vals = [row[c] for c in norm_cols] + [row[norm_cols[0]]]
    ax.plot(angles, vals, "o-", linewidth=2.5, color=COLORS[db], label=db)
    ax.fill(angles, vals, alpha=0.12, color=COLORS[db])

ax.set_thetagrids(np.degrees(angles[:-1]), labels, fontsize=10)
ax.set_ylim(0, 1)
ax.set_yticks([0.25, 0.5, 0.75, 1.0])
ax.set_yticklabels(["0.25", "0.50", "0.75", "1.00"], fontsize=8)
ax.yaxis.grid(True, linestyle="--", alpha=0.4)
ax.xaxis.grid(True, linestyle="--", alpha=0.4)
ax.set_title(
    "Análise Radar — Perfil Multidimensional\n(S1 e S2, valores normalizados 0–1)",
    pad=20, fontsize=12, fontweight="bold"
)
ax.legend(loc="upper right", bbox_to_anchor=(1.35, 1.15), fontsize=11)

plt.tight_layout()
plt.savefig(WORK_DIR / "fig5_radar.png", bbox_inches="tight", dpi=200)
plt.show()
print("✅ Figura 5 (Radar) salva.")

## 9. Resultados Detalhados por Índice (FAISS e Milvus)

In [ ]:
def expand_all_configs(all_results: Dict, db_name: str) -> pd.DataFrame:
    """Expande todos os índices de um banco para análise interna comparativa."""
    rows = []
    for scenario, configs in all_results.items():
        for idx_name, metrics in configs.items():
            rows.append({
                "Banco":          db_name,
                "Cenário":        scenario,
                "Índice":         idx_name,
                "Indexação (s)":  round(metrics.get("index_time_s", np.nan), 2),
                "RAM (MB)":       round(metrics.get("ram_delta_mb", np.nan), 1),
                "Latência p50":   round(metrics.get("p50_ms", np.nan), 3),
                "Latência p99":   round(metrics.get("p99_ms", np.nan), 3),
                "QPS":            round(metrics.get("qps", np.nan), 1),
                "Recall@1":       round(metrics.get("recall@1", np.nan), 4),
                "Recall@5":       round(metrics.get("recall@5", np.nan), 4),
                "Recall@10":      round(metrics.get("recall@10", np.nan), 4),
            })
    return pd.DataFrame(rows)


df_faiss_full  = expand_all_configs(faiss_results,  "FAISS")
df_milvus_full = expand_all_configs(milvus_results, "Milvus")

print("\n=== FAISS — Todas as configurações de índice ===")
display(df_faiss_full.set_index(["Cenário", "Índice"]))

print("\n=== Milvus — Todas as configurações de índice ===")
display(df_milvus_full.set_index(["Cenário", "Índice"]))

## 10. Relatório Resumo — Análise dos Achados

In [ ]:
print("=" * 70)
print("   RELATÓRIO DE BENCHMARK — FAISS × MILVUS × CHROMA em PIPELINES RAG")
print("=" * 70)

for scenario in ["S1_10k", "S2_100k", "S3_1M"]:
    n_vecs = int(scenario.split("_")[1].replace("k","000").replace("M","000000").replace("1000000","1000000"))
    subset = df[df["Cenário"] == scenario]
    print(f"\n{'─'*60}")
    print(f" Cenário {scenario} ({n_vecs:,} vetores)")
    print(f"{'─'*60}")

    for _, row in subset.iterrows():
        print(f"\n  [{row['Banco']}] ({row['Índice']})")
        print(f"    Indexação:    {row['Indexação (s)']:>8.2f} s")
        print(f"    RAM (pico):   {row['RAM (MB)']:>8.1f} MB")
        print(f"    Disco:        {row['Disco (MB)']:>8.1f} MB")
        print(f"    Latência p50: {row['Latência p50 (ms)']:>8.3f} ms")
        print(f"    Latência p99: {row['Latência p99 (ms)']:>8.3f} ms")
        print(f"    QPS:          {row['QPS']:>8.1f}")
        print(f"    Recall@1:     {row['Recall@1']:>8.4f}")
        print(f"    Recall@5:     {row['Recall@5']:>8.4f}")
        print(f"    Recall@10:    {row['Recall@10']:>8.4f}")

print(f"\n{'='*70}")
print(" ANÁLISE COMPARATIVA — PRINCIPAIS ACHADOS")
print(f"{'='*70}")

# Melhor banco por métrica (S2 como referência de produção)
s2 = df[df["Cenário"] == "S2_100k"]
if not s2.empty:
    best_qps    = s2.loc[s2["QPS"].idxmax(), "Banco"]
    best_recall = s2.loc[s2["Recall@10"].idxmax(), "Banco"]
    best_lat    = s2.loc[s2["Latência p50 (ms)"].idxmin(), "Banco"]
    best_ram    = s2.loc[s2["RAM (MB)"].idxmin(), "Banco"]

    print(f"\n  [S2 — 100k vetores, índice ANN]")
    print(f"    Maior throughput (QPS):    {best_qps}")
    print(f"    Maior Recall@10:           {best_recall}")
    print(f"    Menor latência p50:        {best_lat}")
    print(f"    Menor consumo de RAM:      {best_ram}")

print(f"\n  Arquivos de saída:")
for f in sorted(WORK_DIR.glob("fig*.png")):
    print(f"    {f}")

# Salva CSV dos resultados
df.to_csv(WORK_DIR / "resultados_completos.csv", index=False)
df_faiss_full.to_csv(WORK_DIR / "resultados_faiss_detalhado.csv", index=False)
df_milvus_full.to_csv(WORK_DIR / "resultados_milvus_detalhado.csv", index=False)
print(f"\n  CSV exportado: {WORK_DIR / 'resultados_completos.csv'}")
print(f"{'='*70}")

---

## Apêndice — Informações do Ambiente de Execução

Para reprodutibilidade, registrar as informações do ambiente é prática obrigatória em experimentos computacionais.

In [ ]:
import platform
import sys

print("=" * 55)
print("  AMBIENTE DE EXECUÇÃO (para seção de Metodologia)")
print("=" * 55)
print(f"  Sistema Operacional: {platform.system()} {platform.release()}")
print(f"  Arquitetura:         {platform.machine()}")
print(f"  Python:              {sys.version.split()[0]}")
print(f"  CPU:                 {psutil.cpu_count(logical=False)} físicos / {psutil.cpu_count()} lógicos")

ram_gb = psutil.virtual_memory().total / (1024 ** 3)
print(f"  RAM Total:           {ram_gb:.1f} GB")

print(f"\n  Versões das bibliotecas principais:")
import faiss, chromadb, pymilvus, sentence_transformers
print(f"    faiss-cpu:            {faiss.__version__}")
print(f"    chromadb:             {chromadb.__version__}")
print(f"    pymilvus:             {pymilvus.__version__}")
print(f"    sentence-transformers:{sentence_transformers.__version__}")
print(f"    numpy:                {np.__version__}")
print(f"    pandas:               {pd.__version__}")
print(f"\n  Seed de reprodutibilidade: {SEED}")
print(f"  Dimensão dos embeddings:   {EMBEDDING_DIM}")
print(f"  Modelo (S1):               sentence-transformers/all-MiniLM-L6-v2")
print(f"  Métrica de similaridade:   Inner Product (cosine em vetores normalizados)")
print("=" * 55)